In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔐 Cloud-Native Security Guide

**Implementing security throughout cloud-native architectures with zero trust, container security, and supply chain protection**

This guide covers:
- Zero trust security model
- Container and image security
- Secrets management
- Supply chain security
- Network policies and segmentation
- Workload identity
- Runtime security
- Vulnerability scanning and patching
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Zero Trust Architecture

### Zero Trust Principles
```python
from dataclasses import dataclass
from typing import Dict, List, Optional, Set
from enum import Enum
from datetime import datetime

class TrustLevel(Enum):
    """Trust evaluation levels"""
    UNTRUSTED = 0
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    VERIFIED = 4

@dataclass
class SecurityContext:
    """Context for trust evaluation"""
    user_id: str
    service_id: str
    action: str
    resource: str
    timestamp: datetime
    ip_address: str
    device_posture: Dict
    location: str
    endpoint_protection: bool
    mfa_verified: bool
    certificate_valid: bool

class ZeroTrustEvaluator:
    """Evaluate trust for every request"""
    
    def __init__(self):
        self.policies: List[Dict] = []
        self.risk_scores: Dict[str, float] = {}
    
    def add_policy(self, policy: Dict):
        """Add trust policy"""
        self.policies.append(policy)
    
    def evaluate_trust(self, context: SecurityContext) -> TrustLevel:
        """Evaluate trust level for request"""
        
        trust_score = 0.0
        max_score = 0.0
        
        # Check MFA
        if context.mfa_verified:
            trust_score += 2
        max_score += 2
        
        # Check endpoint security
        if context.endpoint_protection:
            trust_score += 2
        max_score += 2
        
        # Check certificate validity
        if context.certificate_valid:
            trust_score += 2
        max_score += 2
        
        # Check location risk
        if context.location in ["office"]:
            trust_score += 1
        max_score += 1
        
        # Normalize score
        normalized_score = trust_score / max_score if max_score > 0 else 0
        
        if normalized_score >= 0.9:
            return TrustLevel.VERIFIED
        elif normalized_score >= 0.7:
            return TrustLevel.HIGH
        elif normalized_score >= 0.5:
            return TrustLevel.MEDIUM
        elif normalized_score >= 0.2:
            return TrustLevel.LOW
        else:
            return TrustLevel.UNTRUSTED
    
    def should_allow_access(self, context: SecurityContext) -> bool:
        """Determine if access should be allowed"""
        trust_level = self.evaluate_trust(context)
        
        # Require at least MEDIUM trust
        return trust_level.value >= TrustLevel.MEDIUM.value
    
    def apply_adaptive_controls(self, context: SecurityContext) -> Dict:
        """Apply controls based on trust level"""
        trust_level = self.evaluate_trust(context)
        
        controls = {
            'require_mfa': trust_level.value < TrustLevel.HIGH.value,
            'require_vpn': trust_level.value < TrustLevel.MEDIUM.value,
            'require_device_compliance': True,
            'rate_limit': 1000,
            'timeout_seconds': 3600
        }
        
        if trust_level == TrustLevel.UNTRUSTED:
            controls['allow_access'] = False
            controls['alert_security_team'] = True
        elif trust_level == TrustLevel.LOW:
            controls['rate_limit'] = 100
            controls['require_mfa'] = True
        elif trust_level == TrustLevel.HIGH:
            controls['rate_limit'] = 5000
        
        return controls

# Usage
evaluator = ZeroTrustEvaluator()

context = SecurityContext(
    user_id="user123",
    service_id="api-gateway",
    action="read",
    resource="/api/users",
    timestamp=datetime.now(),
    ip_address="203.0.113.42",
    device_posture={"compliant": True, "updated": True},
    location="office",
    endpoint_protection=True,
    mfa_verified=True,
    certificate_valid=True
)

trust_level = evaluator.evaluate_trust(context)
can_access = evaluator.should_allow_access(context)
controls = evaluator.apply_adaptive_controls(context)
```

### Identity & Access Management
```python
class WorkloadIdentity:
    """Cryptographic identity for workloads"""
    
    def __init__(self, workload_name: str):
        self.workload_name = workload_name
        self.certificate: Optional[str] = None
        self.key: Optional[str] = None
        self.issuer: str = "internal-ca"
        self.expiry: Optional[datetime] = None
        self.service_account: str = f"{workload_name}@workload.iam"
    
    def is_valid(self) -> bool:
        """Check if identity is valid"""
        if not self.certificate or not self.key:
            return False
        
        if self.expiry and datetime.now() > self.expiry:
            return False
        
        return True
    
    def request_short_lived_credential(self, ttl_hours: int = 1) -> Dict:
        """Request short-lived credential"""
        return {
            'workload': self.workload_name,
            'service_account': self.service_account,
            'token_ttl_hours': ttl_hours,
            'expires_at': datetime.now()  # Would add ttl
        }

class RBAC:
    """Role-based access control"""
    
    def __init__(self):
        self.roles: Dict[str, Set[str]] = {}  # role -> permissions
        self.bindings: Dict[str, Set[str]] = {}  # principal -> roles
    
    def create_role(self, role_name: str, permissions: List[str]):
        """Create role with permissions"""
        self.roles[role_name] = set(permissions)
    
    def bind_role(self, principal: str, role: str):
        """Bind role to principal"""
        if principal not in self.bindings:
            self.bindings[principal] = set()
        
        self.bindings[principal].add(role)
    
    def get_permissions(self, principal: str) -> Set[str]:
        """Get effective permissions for principal"""
        permissions = set()
        
        roles = self.bindings.get(principal, set())
        
        for role in roles:
            permissions.update(self.roles.get(role, set()))
        
        return permissions
    
    def can_perform(self, principal: str, action: str) -> bool:
        """Check if principal can perform action"""
        permissions = self.get_permissions(principal)
        return action in permissions
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Container & Image Security

### Vulnerability Scanning
```python
class Vulnerability:
    """Security vulnerability finding"""
    
    def __init__(self, cve_id: str, severity: str):
        self.cve_id = cve_id
        self.severity = severity  # critical, high, medium, low
        self.component: str = ""
        self.version: str = ""
        self.fix_version: Optional[str] = None
        self.description: str = ""
        self.cvss_score: float = 0.0
    
    def is_critical(self) -> bool:
        return self.severity in ["critical", "high"]
    
    def has_fix(self) -> bool:
        return self.fix_version is not None

class ImageScanner:
    """Scan container images for vulnerabilities"""
    
    def __init__(self):
        self.scan_results: Dict[str, List[Vulnerability]] = {}
    
    def scan_image(self, image_name: str) -> List[Vulnerability]:
        """Scan image for vulnerabilities"""
        
        # Simulate scanning (real implementation would use tools like Trivy)
        vulnerabilities = [
            Vulnerability("CVE-2023-1234", "high"),
            Vulnerability("CVE-2023-5678", "medium"),
        ]
        
        self.scan_results[image_name] = vulnerabilities
        return vulnerabilities
    
    def get_severity_summary(self, image_name: str) -> Dict[str, int]:
        """Get vulnerability count by severity"""
        vulns = self.scan_results.get(image_name, [])
        
        summary = {
            'critical': 0,
            'high': 0,
            'medium': 0,
            'low': 0
        }
        
        for vuln in vulns:
            summary[vuln.severity] += 1
        
        return summary
    
    def should_block_image(self, image_name: str, allow_high_severity: bool = False) -> bool:
        """Determine if image should be blocked"""
        summary = self.get_severity_summary(image_name)
        
        if summary['critical'] > 0:
            return True
        
        if not allow_high_severity and summary['high'] > 0:
            return True
        
        return False

class RuntimeSecurityPolicy:
    """Define runtime security policies"""
    
    def __init__(self):
        self.policies: List[Dict] = []
    
    def add_policy(self, name: str, rules: Dict):
        """Add security policy"""
        self.policies.append({
            'name': name,
            'rules': rules,
            'enabled': True
        })
    
    def add_syscall_policy(self, syscalls: List[str], action: str = "block"):
        """Restrict syscalls"""
        self.add_policy("syscall_restriction", {
            'blocked_syscalls': syscalls,
            'action': action
        })
    
    def add_capability_policy(self, capabilities: List[str]):
        """Drop Linux capabilities"""
        self.add_policy("drop_capabilities", {
            'drop': capabilities
        })
    
    def get_pod_security_policy(self) -> Dict:
        """Generate pod security policy"""
        return {
            'privileged': False,
            'capabilities': {
                'drop': ['ALL'],
                'add': ['NET_BIND_SERVICE']
            },
            'readOnlyRootFilesystem': True,
            'runAsNonRoot': True,
            'runAsUser': 1000
        }
```

### Secrets Management
```python
class SecretsManager:
    """Manage application secrets securely"""
    
    def __init__(self):
        self.secrets: Dict[str, Dict] = {}
        self.rotation_policies: Dict[str, Dict] = {}
    
    def create_secret(self, name: str, value: str, secret_type: str = "generic"):
        """Create secret"""
        self.secrets[name] = {
            'value': value,
            'type': secret_type,
            'created_at': datetime.now(),
            'rotation_needed': False
        }
    
    def rotate_secret(self, name: str, new_value: str):
        """Rotate secret"""
        if name not in self.secrets:
            raise ValueError(f"Secret {name} not found")
        
        old_value = self.secrets[name]['value']
        self.secrets[name]['value'] = new_value
        self.secrets[name]['last_rotated'] = datetime.now()
        self.secrets[name]['previous_value'] = old_value
    
    def set_rotation_policy(self, secret_name: str, rotation_days: int):
        """Set automatic rotation policy"""
        self.rotation_policies[secret_name] = {
            'rotation_days': rotation_days,
            'enabled': True
        }
    
    def get_rotation_due(self) -> List[str]:
        """Get secrets needing rotation"""
        rotation_due = []
        
        for secret_name, policy in self.rotation_policies.items():
            if not policy['enabled']:
                continue
            
            secret = self.secrets.get(secret_name)
            if not secret:
                continue
            
            last_rotated = secret.get('last_rotated', secret.get('created_at'))
            
            if (datetime.now() - last_rotated).days > policy['rotation_days']:
                rotation_due.append(secret_name)
        
        return rotation_due
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Supply Chain Security

### Artifact Signing
```python
class SignedArtifact:
    """Cryptographically signed artifact"""
    
    def __init__(self, artifact_id: str):
        self.artifact_id = artifact_id
        self.signature: Optional[str] = None
        self.signer: Optional[str] = None
        self.signed_at: Optional[datetime] = None
        self.certificate_chain: List[str] = []
    
    def sign(self, private_key: str, signer_identity: str):
        """Sign artifact"""
        # Simplified - real implementation uses cryptographic signing
        self.signature = f"sig_{self.artifact_id}_{hash(private_key)}"
        self.signer = signer_identity
        self.signed_at = datetime.now()
    
    def verify_signature(self, public_key: str) -> bool:
        """Verify artifact signature"""
        if not self.signature:
            return False
        
        # Simplified verification
        return self.signature.startswith("sig_")

class ArtifactRegistry:
    """Secure artifact storage and registry"""
    
    def __init__(self):
        self.artifacts: Dict[str, SignedArtifact] = {}
        self.policies: List[Dict] = []
    
    def register_artifact(self, artifact_id: str, signed_artifact: SignedArtifact):
        """Register signed artifact"""
        self.artifacts[artifact_id] = signed_artifact
    
    def require_attestation(self, attestation_type: str):
        """Require attestation for artifacts"""
        self.policies.append({
            'type': 'attestation_required',
            'attestation_type': attestation_type
        })
    
    def get_provenance(self, artifact_id: str) -> Dict:
        """Get artifact provenance"""
        artifact = self.artifacts.get(artifact_id)
        
        if not artifact:
            return {}
        
        return {
            'artifact_id': artifact_id,
            'signed_by': artifact.signer,
            'signed_at': artifact.signed_at.isoformat() if artifact.signed_at else None,
            'signature_valid': artifact.verify_signature("public_key")
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Cloud-Native Security Checklist

✅ **Zero Trust Implementation**
- [ ] Trust evaluation framework
- [ ] Continuous authentication
- [ ] Contextual authorization
- [ ] Least privilege enforced
- [ ] Adaptive controls working

✅ **Container Security**
- [ ] Image scanning enabled
- [ ] Vulnerabilities remediated
- [ ] Runtime policies enforced
- [ ] Capabilities dropped
- [ ] Root filesystem read-only

✅ **Secrets Management**
- [ ] Secrets encrypted at rest
- [ ] Secrets encrypted in transit
- [ ] Rotation policies active
- [ ] No hardcoded secrets
- [ ] Access logging enabled

✅ **Supply Chain Security**
- [ ] Artifacts signed
- [ ] Signatures verified
- [ ] Attestations required
- [ ] Provenance tracked
- [ ] Build logs audited

✅ **Compliance**
- [ ] Compliance scanning active
- [ ] Policy violations detected
- [ ] Audit trails maintained
- [ ] Alerts configured
- [ ] Regular audits performed
</VSCode.Cell>
```